# BC Training — Google Colab

Запускает supervised обучение политики на собранном датасете.

**Перед запуском:** загрузи на Google Drive три файла:
- `bc_dataset.npz`
- `bc_dataset_obs.dat` (~7.2 ГБ)
- `bc_dataset_act.dat` (~4 МБ)

Положи их в папку `RL_practice/datasets/` на Drive.

In [ ]:
# Монтируем Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Клонируем репо и устанавливаем зависимости
!git clone https://github.com/Andrew82mm/RL_practice.git /content/RL_practice
%cd /content/RL_practice
!pip install -q sb3-contrib gymnasium torch scipy tqdm

In [ ]:
# Проверяем GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не найден — смени Runtime на GPU!'}")

In [ ]:
import os, shutil

DRIVE_DATASET_DIR = "/content/drive/MyDrive/RL_practice/datasets"
LOCAL_DIR = "/content/dataset"
os.makedirs(LOCAL_DIR, exist_ok=True)

# Копируем данные с Drive на локальный диск Colab
# Это важно: DataLoader с shuffle=True делает случайные чтения —
# с Drive это будет в 10-50x медленнее чем с локального SSD
print("Копируем obs.dat (~7.2 ГБ) с Drive на локальный диск...")
print("(займёт 5-15 минут)")

for fname in ["bc_dataset.npz", "bc_dataset_obs.dat", "bc_dataset_act.dat"]:
    src = f"{DRIVE_DATASET_DIR}/{fname}"
    dst = f"{LOCAL_DIR}/{fname}"
    if not os.path.exists(dst):
        print(f"  Копирую {fname}...")
        shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / 1024**2
    print(f"  {fname}: {size_mb:.0f} MB — OK")

In [ ]:
import numpy as np

# Патчим .npz: обновляем пути к .dat файлам на локальные
meta = np.load(f"{LOCAL_DIR}/bc_dataset.npz", allow_pickle=True)
np.savez(
    f"{LOCAL_DIR}/bc_dataset.npz",
    n_samples=meta["n_samples"],
    obs_shape=meta["obs_shape"],
    obs_path=np.array(f"{LOCAL_DIR}/bc_dataset_obs.dat"),
    act_path=np.array(f"{LOCAL_DIR}/bc_dataset_act.dat"),
)
print(f"Датасет готов: {int(meta['n_samples']):,} пар, obs shape {tuple(meta['obs_shape'])}")

In [ ]:
# Запускаем BC обучение
!python training/bc_train.py \
    --only-train \
    --dataset-path "{LOCAL_DIR}/bc_dataset.npz" \
    --pretrained-path "/content/drive/MyDrive/RL_practice/models/bc_pretrained"

In [ ]:
# Проверяем что модель сохранилась на Drive
model_path = "/content/drive/MyDrive/RL_practice/models/bc_pretrained.zip"
size_mb = os.path.getsize(model_path) / 1024**2
print(f"Модель сохранена: {model_path} ({size_mb:.1f} MB)")
print("Готово! Скачай bc_pretrained.zip с Drive.")